## Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
data = pd.read_csv('data/taxi.csv', parse_dates=[0], index_col=0)
data.sort_index(inplace=True)
data.head()

In [ ]:
data.info()

- The dataset contains only timestamps and the number of orders. There are no missing values.

In [ ]:
data_1h = data.resample('1h').sum()

In [ ]:
data_1h.head()

In [ ]:
data_1h.info()

- Orders are aggregated into hourly totals to match the prediction target.

## Exploratory Analysis

In [ ]:
data_1h.describe()

In [ ]:
data_1h['rolling_mean'] = data_1h['num_orders'].rolling(24).mean()

data_1h[['num_orders', 'rolling_mean']].plot(figsize=(12, 5))
plt.xlabel('Date')
plt.ylabel('Number of orders')
plt.show()

In [ ]:
decomposed = seasonal_decompose(
    data_1h['num_orders'],
    period=24
)

plt.figure(figsize=(7, 9))

plt.subplot(311)
decomposed.trend.plot(ax=plt.gca())
plt.title('Trend')

plt.subplot(312)
decomposed.seasonal.plot(ax=plt.gca())
plt.title('Seasonality')

plt.subplot(313)
decomposed.resid.plot(ax=plt.gca())
plt.title('Residuals')

plt.tight_layout()

- Taxi demand averages around 84 orders per hour, with occasional spikes.
- The rolling average shows a gradual increase over the period.
- The decomposition suggests a daily seasonal pattern, alongside some large residual spikes.

## Model Training

In [ ]:
def make_features(df_1h, max_lag, rolling_mean_size):
    df_1h['year'] = df_1h.index.year
    df_1h['month'] = df_1h.index.month
    df_1h['day'] = df_1h.index.day
    df_1h['dayofweek'] = df_1h.index.dayofweek
    df_1h['hour'] = df_1h.index.hour

    for lag in range(1, max_lag + 1):
        df_1h['lag_{}'.format(lag)] = df_1h['num_orders'].shift(lag)

    df_1h['rolling_mean'] = (
        df_1h['num_orders']
        .shift()
        .rolling(rolling_mean_size)
        .mean()
    )

In [ ]:
make_features(data_1h, 168, 24)

In [ ]:
train_full, test = train_test_split(
    data_1h,
    shuffle=False,
    test_size=0.1
)

train_full = train_full.dropna()

train, valid = train_test_split(
    train_full,
    shuffle=False,
    test_size=0.2
)

print('Train:', train.shape)
print('Validation:', valid.shape)
print('Test:', test.shape)

In [ ]:
features_train = train.drop(['num_orders'], axis=1)
target_train = train['num_orders']

features_valid = valid.drop(['num_orders'], axis=1)
target_valid = valid['num_orders']

features_test = test.drop(['num_orders'], axis=1)
target_test = test['num_orders']

print('Missing values in train:', features_train.isna().sum().sum())
print('Missing values in validation:', features_valid.isna().sum().sum())
print('Missing values in test:', features_test.isna().sum().sum())

In [ ]:
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

### Linear Regression

In [ ]:
model_lr = LinearRegression()
model_lr.fit(features_train, target_train)

predictions_lr = model_lr.predict(features_valid)
rmse_lr = calculate_rmse(target_valid, predictions_lr)

print('Linear Regression')
print('RMSE:', rmse_lr)

### Decision Tree

In [ ]:
tree_results = []

for depth in [3, 5, 10, 15]:
    model_tree = DecisionTreeRegressor(
        max_depth=depth,
        random_state=12345
    )

    model_tree.fit(features_train, target_train)
    predictions_tree = model_tree.predict(features_valid)
    rmse_tree = calculate_rmse(target_valid, predictions_tree)

    tree_results.append((depth, rmse_tree))

    print('Decision Tree:', 'max_depth:', depth, 'RMSE:', rmse_tree)

### Random Forest

In [ ]:
forest_results = []

for depth in [10, 15, 20, 40]:
    model_forest = RandomForestRegressor(
        n_estimators=100,
        max_depth=depth,
        random_state=12345,
        n_jobs=-1
    )

    model_forest.fit(features_train, target_train)
    predictions_forest = model_forest.predict(features_valid)
    rmse_forest = calculate_rmse(target_valid, predictions_forest)

    forest_results.append((depth, rmse_forest))

    print('Random Forest:', 'max_depth:', depth, 'RMSE:', rmse_forest)

In [ ]:
best_tree_depth, best_tree_rmse = min(tree_results, key=lambda x: x[1])
best_forest_depth, best_forest_rmse = min(forest_results, key=lambda x: x[1])

results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest'],
    'Validation RMSE': [rmse_lr, best_tree_rmse, best_forest_rmse],
    'Best hyperparameter': [
        '-',
        f'max_depth={best_tree_depth}',
        f'max_depth={best_forest_depth}'
    ]
})

results.sort_values('Validation RMSE')

In [ ]:
best_model_name = results.sort_values('Validation RMSE').iloc[0]['Model']
print('Best model:', best_model_name)

- Linear Regression achieved the lowest validation RMSE (**27.13**).

## Final Evaluation

In [ ]:
train_final = pd.concat([train, valid])

features_train_final = train_final.drop(['num_orders'], axis=1)
target_train_final = train_final['num_orders']

In [ ]:
if best_model_name == 'Linear Regression':
    final_model = LinearRegression()
elif best_model_name == 'Decision Tree':
    final_model = DecisionTreeRegressor(
        max_depth=best_tree_depth,
        random_state=12345
    )
else:
    final_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=best_forest_depth,
        random_state=12345,
        n_jobs=-1
    )

final_model.fit(features_train_final, target_train_final)
predictions_test = final_model.predict(features_test)
rmse_test = calculate_rmse(target_test, predictions_test)

print('Best model:', best_model_name)
print('Test RMSE:', rmse_test)

- The selected Linear Regression model was retrained on the combined training and validation sets.
- It achieved a **test RMSE of 35.46**, meeting the target of less than 48.
- Performance was weaker on the test period than on validation, so forecast quality may vary over time.